In [ ]:
import numpy as np
import sklearn
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn import pipeline

# Loading the Raw Data Needed to Build a Matrix Like:
$$
R_{ij}
$$
Where:
1. $i$ = A User
2. $j$ = A Book
3. $Rij$ = The rating user $i$ gave book $j$

In [ ]:
data_books = pd.read_csv('/content/drive/MyDrive/Datasets/Books.csv', low_memory=False, encoding="latin-1") # Encode in Latin 1 and set low_memory to false for special characters and mixed column types
data_ratings = pd.read_csv('/content/drive/MyDrive/Datasets/Ratings.csv')

In [ ]:

data_ratings = data_ratings[data_ratings["Book-Rating"] != 0].copy()


In [ ]:
top_users = data_ratings['User-ID'].value_counts().head(1000).index # Set-like list of selected users
top_books = data_ratings['ISBN'].value_counts().head(2000).index # Set-like list of selected Books

data_ratings = data_ratings[
    data_ratings['User-ID'].isin(top_users) &
    data_ratings['ISBN'].isin(top_books)
].copy()

print("Filtered ratings shape:", data_ratings.shape)
print("Unique books:", data_ratings['ISBN'].nunique())
print("Unique users:", data_ratings['User-ID'].nunique())

Filtered ratings shape: (25524, 3)
Unique books: 1998
Unique users: 966


In [ ]:
# 2. Setup ID Mappings
books_uniques = np.unique(data_ratings.ISBN.values)
users_uniques = np.unique(data_ratings['User-ID'].values)

books_dict_real_ref = {res: idx for idx, res in enumerate(books_uniques)}
books_dict_ref_real = {idx: res for idx, res in enumerate(books_uniques)}
users_dict_real_ref = {res: idx for idx, res in enumerate(users_uniques)}


In [ ]:
ratings_mat = np.zeros(shape=(len(books_uniques), len(users_uniques)), dtype=np.float32)


In [ ]:
# Rows = Books, Cols = Users
for user_id, isbn, rating in data_ratings[['User-ID', 'ISBN', 'Book-Rating']].itertuples(index=False, name=None):
    ref_u = users_dict_real_ref[user_id]
    ref_b = books_dict_real_ref[isbn]
    ratings_mat[ref_b, ref_u] = rating


In [ ]:
# Means Centering (Fixing "0" Rating)
book_means = np.array(
    [np.mean(row[row > 0]) if np.any(row > 0)
    else 0 for row in ratings_mat],
    dtype=np.float32
)
ratings_mat_centered = ratings_mat - book_means.reshape(-1, 1)
ratings_mat_centered[ratings_mat == 0] = 0


In [ ]:
U, S, V = np.linalg.svd(ratings_mat_centered, full_matrices=False)
print("SVD singular values:", S)
print("U matrix:", U)
print("V^T matrix:", V)

SVD singular values: [5.46894112e+01 3.37750969e+01 3.17038593e+01 2.70417309e+01
 2.49611835e+01 2.45015926e+01 2.37209225e+01 2.30281067e+01
 2.25566044e+01 2.19270382e+01 2.15928421e+01 2.13051491e+01
 2.11431351e+01 2.09063282e+01 2.07925377e+01 2.02952118e+01
 2.01731815e+01 1.99271717e+01 1.97494869e+01 1.94555550e+01
 1.92647934e+01 1.92094574e+01 1.90783443e+01 1.88060818e+01
 1.86502838e+01 1.84686146e+01 1.82899456e+01 1.82360783e+01
 1.81730385e+01 1.80700512e+01 1.78850212e+01 1.78216763e+01
 1.77643871e+01 1.75609570e+01 1.74270306e+01 1.73040180e+01
 1.72623444e+01 1.71493855e+01 1.70413857e+01 1.69775562e+01
 1.68157921e+01 1.67012596e+01 1.66221313e+01 1.64963398e+01
 1.63718987e+01 1.62921066e+01 1.62304916e+01 1.61941414e+01
 1.59049759e+01 1.58588266e+01 1.57330179e+01 1.56714296e+01
 1.56269779e+01 1.54987097e+01 1.54465485e+01 1.53307362e+01
 1.52470722e+01 1.52076416e+01 1.51393776e+01 1.50889492e+01
 1.50546608e+01 1.49085789e+01 1.48731699e+01 1.48629541e+01
 1.

In [ ]:
S_square = np.square(S)
p = S_square / np.sum(S_square)
svd_entropy = -np.sum(p[p > 0] * np.log2(p[p > 0]))

print("SVD Entropy:", svd_entropy)

energy = np.sum(S_square[:2]) / np.sum(S_square)
print(energy)

SVD Entropy: 8.660443
0.06341665


In [ ]:
k = 50
book_latent = U[:, :k] * S[:k]

In [ ]:
book_variability = np.array(
    [np.var(row[row > 0]) if np.any(row > 0) and len(row[row > 0]) > 1 else 0
     for row in ratings_mat],
    dtype=np.float32
)

In [ ]:
# Weight Recommendation by Information Richness
# Combining the Latent Similarity with Entropy so that highly informative books influence recommendations score
def get_recommendations_info_weighted(latent_matrix, book_real_id, book_variability, top_n=10):
    ref_id = books_dict_real_ref[book_real_id]
    book_vector = latent_matrix[ref_id, :].reshape(1, -1)

    scores = cosine_similarity(book_vector, latent_matrix).flatten()
    weights = 1 + book_variability / (book_variability.max() + 1e-8)
    weighted_scores = scores * weights
    top_indices = np.argsort(weighted_scores)[::-1]

    return top_indices[1:top_n+1]



In [ ]:
k = 50
search_isbn = books_uniques[0]
reduced_U = U[:, :k]


book_entropies = np.array([np.var(row[row > 0]) if np.any(row > 0) and len(row[row > 0]) > 1 else 0 for row in ratings_mat], dtype=np.float32)

top_indices = get_recommendations_info_weighted(reduced_U, search_isbn, book_variability, top_n=10)

In [ ]:
# Print readable titles
search_title = data_books.loc[data_books['ISBN'] == search_isbn, 'Book-Title']
search_title = search_title.values[0] if len(search_title) else f"(title not found for ISBN {search_isbn})"
print(f"Books similar to: {search_title}\n" + "-"*30)

for idx in top_indices:
    real_isbn = books_dict_ref_real[idx]
    title = data_books.loc[data_books['ISBN'] == real_isbn, 'Book-Title']
    title = title.values[0] if len(title) else "(title missing)"
    print(f"ISBN {real_isbn}: {title}")

Books similar to: Angelas Ashes
------------------------------
ISBN 0385482388: The Mistress of Spices
ISBN 1853260002: Pride &amp; Prejudice (Wordsworth Classics)
ISBN 0375758992: Don't Let's Go to the Dogs Tonight : An African Childhood
ISBN 0425188787: Hunting Season (Anna Pigeon Novels (Paperback))
ISBN 0385474547: Things Fall Apart : A Novel
ISBN 0786866276: Mother of Pearl
ISBN 0446603716: Hide &amp; Seek
ISBN 0399145672: Big Trouble
ISBN 0440221315: The Gift
ISBN 0385413041: Beach Music
